# Predictive Analysis of Road Accident Hotspots in Bangalore
### Using Machine Learning and Explainable AI
**Authors:** Guddanti Nikhil Srinivas, Naragani Lakshmi Venkata Manohar  
**Alliance University, Bengaluru** | Dataset: Open City India (2018–2025)

## ⚙️ Setup

In [ ]:
import os
from pathlib import Path

# Always sets correct working directory regardless of where notebook runs
project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
os.chdir(project_root)
print(f'Working directory set to: {project_root}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split

## 📊 Exploratory Data Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Ensure numpy is imported for np.nan and np.select

# Load wide dataset

df = pd.read_csv("data/final_perfect_dataset.csv")

# Define year_cols for consistency
year_cols = ["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"]
# Replace 0 with NaN
df[year_cols] = df[year_cols].replace(0, np.nan)
# Remove stations with no data at all
df = df[df[year_cols].notna().sum(axis=1) > 0]

# Convert to long format
df_long = df.melt(
    id_vars=["Station", "Zone", "Latitude", "Longitude"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Total"
)

df_long["Year"] = df_long["Year"].str.replace("Y","").astype(int)

# Style
sns.set(style="whitegrid")

#YEAR-WISE TREND
yearly = df_long.groupby("Year")["Total"].sum()

plt.figure(figsize=(6,4))
yearly.plot(marker='o')

plt.title("Year-wise Accident Trend (2018–2025)")
plt.xlabel("Year")
plt.ylabel("Total Accidents")

plt.tight_layout()
plt.savefig("trend_graph.png")
plt.show()
print("\n")

#TOP 10 ACCIDENT HOTSPOTS
top = df_long.groupby("Station")["Total"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(7,4))
top.plot(kind="bar")

plt.title("Top 10 Accident Hotspots")
plt.xlabel("Station")
plt.ylabel("Total Accidents")

plt.tight_layout()
plt.savefig("top_hotspots.png")
plt.show()
print("\n")

#ZONE-WISE ANALYSIS
zone = df_long.groupby("Zone")["Total"].sum()

plt.figure(figsize=(6,4))
zone.plot(kind="bar")

plt.title("Zone-wise Accident Distribution")
plt.xlabel("Zone")
plt.ylabel("Total Accidents")

plt.tight_layout()
plt.savefig("zone_analysis.png")
plt.show()
print("\n")

#RISK LEVEL DISTRIBUTION

# Calculate 'Average' column for df before using it in conditions
df["Average"] = df[year_cols].mean(axis=1)

conditions = [
    df["Average"] < 50,
    (df["Average"] >= 50) & (df["Average"] < 100),
    df["Average"] >= 100
]

risk_labels = ["Low Risk", "Medium Risk", "High Risk"]

df["Risk_Level"] = np.select(conditions, risk_labels, default="Unknown")

plt.figure(figsize=(6,4))

sns.countplot(
    data=df,
    x="Risk_Level",
    hue="Risk_Level",
    palette="viridis",
    legend=False
)

plt.title("Accident Risk Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Number of Cases")

plt.tight_layout()
plt.savefig("risk_distribution.png")
plt.show()
print("\n")

#YEAR-WISE ZONE COMPARISON
pivot_zone = df_long.pivot_table(
    values="Total",
    index="Year",
    columns="Zone",
    aggfunc="sum"
)

plt.figure(figsize=(7,5))
pivot_zone.plot()

plt.title("Zone-wise Trend Over Years")
plt.xlabel("Year")
plt.ylabel("Total Accidents")

plt.tight_layout()
plt.savefig("zone_trend.png")
plt.show()
print("\n")

## 🤖 Baseline Model (Random Forest)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


df = pd.read_csv("data/final_perfect_dataset.csv")

# CONVERT TO LONG FORMAT
df_long = df.melt(
    id_vars=["Station", "Zone", "Latitude", "Longitude"],
    value_vars=["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"],
    var_name="Year",
    value_name="Total"
)

df_long["Year"] = df_long["Year"].str.replace("Y"," ").astype(int)

# FEATURES ENGINEERING

df_long = df_long.sort_values(["Station", "Year"])

df_long["Prev_Year"] = df_long.groupby("Station")["Total"].shift(1)
df_long["Trend"] = df_long["Total"] - df_long["Prev_Year"]
df_long["Rolling_Avg"] = df_long.groupby("Station")["Total"].rolling(2).mean().reset_index(0, drop=True)

df_long = df_long.fillna(0)

# CREATE RISK LEVEL
conditions = [
    (df_long["Total"] < 30),
    (df_long["Total"] < 70),
    (df_long["Total"] >= 70)
]

labels = ["Low Risk", "Medium Risk", "High Risk"]

df_long["Risk_Level"] = np.select(conditions, labels, default="")

# ENCODING
le_station = LabelEncoder()
le_zone = LabelEncoder()
le_risk = LabelEncoder()

df_long["Station_Enc"] = le_station.fit_transform(df_long["Station"])
df_long["Zone_Enc"] = le_zone.fit_transform(df_long["Zone"])
df_long["Risk_Enc"] = le_risk.fit_transform(df_long["Risk_Level"])

# REGRESSION MODEL

X_reg = df_long[[
    "Year",
    "Station_Enc",
    "Zone_Enc",
    "Prev_Year",
    "Trend",
    "Rolling_Avg"
]]

y_reg = df_long["Total"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2)

reg_model = RandomForestRegressor(n_estimators=200)
reg_model.fit(X_train_r, y_train_r)

print("Regression R²:", reg_model.score(X_test_r, y_test_r))

# CLASSIFICATION MODEL

X_clf = X_reg
y_clf = df_long["Risk_Enc"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.2)

clf_model = RandomForestClassifier(n_estimators=200)
clf_model.fit(X_train_c, y_train_c)

print("Classification Accuracy:", clf_model.score(X_test_c, y_test_c))

# STATION LIST

stations = sorted(df["Station"].unique())

print("\n📍 Available Stations:")
for s in stations[:]:
    print("-", s)


# MENU SYSTEM

while True:
    print("\n===== MENU =====")
    print("1. Predict Accident Count")
    print("2. Predict Risk Level")
    print("3. Exit")

    choice = input("Enter your choice: ")

    if choice == "1" or choice == "2":

        station_input = input("Enter Station Name: ")

        if station_input not in stations:
            print("❌ Invalid station!")
            continue

        try:
            year_input = int(input("Enter Year (e.g., 2026): "))
        except:
            print("❌ Invalid year!")
            continue

        # Encode station & zone
        station_enc = le_station.transform([station_input])[0]
        zone_name = df[df["Station"] == station_input]["Zone"].iloc[0]
        zone_enc = le_zone.transform([zone_name])[0]

        # GET PREVIOUS DATA

        station_data = df_long[df_long["Station"] == station_input].sort_values("Year")

        last_row = station_data.iloc[-1]

        prev_year = last_row["Total"]
        trend = last_row["Total"] - station_data.iloc[-2]["Total"] if len(station_data) > 1 else 0
        rolling_avg = station_data["Total"].tail(2).mean()

        # Input for model
        user_data = pd.DataFrame({
            "Year": [year_input],
            "Station_Enc": [station_enc],
            "Zone_Enc": [zone_enc],
            "Prev_Year": [prev_year],
            "Trend": [trend],
            "Rolling_Avg": [rolling_avg]
        })

        # REGRESSION

        if choice == "1":
            pred = reg_model.predict(user_data)

            print("\n✅ Prediction:")
            print(f"Station: {station_input}")
            print(f"Year: {year_input}")
            print(f"Predicted Accidents: {int(pred[0])}")

        # CLASSIFICATION

        elif choice == "2":
            pred = clf_model.predict(user_data)
            risk = le_risk.inverse_transform(pred)[0]

            print("\n✅ Prediction:")
            print(f"Station: {station_input}")
            print(f"Year: {year_input}")
            print(f"Predicted Risk Level: {risk}")

    elif choice == "3":
        print("Exiting...")
        break

    else:
        print("❌ Invalid choice!")

## 📈 Comparative Model Analysis
*Random Forest vs XGBoost vs Decision Tree*

In [ ]:
# ============================================================
# CELL 1 — COMPARATIVE MODEL ANALYSIS
# ============================================================

from xgboost import XGBClassifier, XGBRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_squared_error, mean_absolute_error, r2_score,
    confusion_matrix, classification_report
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── DEFINE ALL 3 MODELS ──────────────────────────────────────

clf_models = {
    "Random Forest":    RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost":          XGBClassifier(n_estimators=200, use_label_encoder=False,
                                      eval_metric="mlogloss", random_state=42),
    "Decision Tree":    DecisionTreeClassifier(random_state=42)
}

reg_models = {
    "Random Forest":    RandomForestRegressor(n_estimators=200, random_state=42),
    "XGBoost":          XGBRegressor(n_estimators=200, random_state=42),
    "Decision Tree":    DecisionTreeRegressor(random_state=42)
}

# ── TRAIN & EVALUATE CLASSIFIERS ─────────────────────────────

clf_results = {}
clf_preds   = {}

print("=" * 55)
print("       CLASSIFICATION MODEL COMPARISON")
print("=" * 55)

for name, model in clf_models.items():
    model.fit(X_train_c, y_train_c)
    y_pred = model.predict(X_test_c)
    clf_preds[name] = y_pred
    clf_results[name] = {
        "Accuracy":  round(accuracy_score(y_test_c, y_pred), 4),
        "Precision": round(precision_score(y_test_c, y_pred, average="weighted", zero_division=0), 4),
        "Recall":    round(recall_score(y_test_c, y_pred, average="weighted"), 4),
        "F1 Score":  round(f1_score(y_test_c, y_pred, average="weighted"), 4),
    }
    print(f"\n🔹 {name}")
    for k, v in clf_results[name].items():
        print(f"   {k:<12}: {v}")

clf_df = pd.DataFrame(clf_results).T
print("\n\nClassification Summary Table:")
print(clf_df.to_string())

# ── TRAIN & EVALUATE REGRESSORS ───────────────────────────────

reg_results = {}
reg_preds   = {}

print("\n\n" + "=" * 55)
print("        REGRESSION MODEL COMPARISON")
print("=" * 55)

for name, model in reg_models.items():
    model.fit(X_train_r, y_train_r)
    y_pred = model.predict(X_test_r)
    reg_preds[name] = y_pred
    reg_results[name] = {
        "R² Score": round(r2_score(y_test_r, y_pred), 4),
        "RMSE":     round(np.sqrt(mean_squared_error(y_test_r, y_pred)), 4),
        "MAE":      round(mean_absolute_error(y_test_r, y_pred), 4),
    }
    print(f"\n🔹 {name}")
    for k, v in reg_results[name].items():
        print(f"   {k:<12}: {v}")

reg_df = pd.DataFrame(reg_results).T
print("\n\nRegression Summary Table:")
print(reg_df.to_string())

## 🎨 Professional Visualizations

In [ ]:
# ============================================================
# CELL 2 — PROFESSIONAL VISUALIZATIONS
# ============================================================

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi":     150,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

PALETTE   = ["#2C6E91", "#E07B3F", "#3BAD6B"]   # blue, orange, green
MODEL_NAMES = list(clf_results.keys())

# ─────────────────────────────────────────────────────────────
# FIG 1 ─ Classification Metrics Grouped Bar Chart
# ─────────────────────────────────────────────────────────────

metrics   = ["Accuracy", "Precision", "Recall", "F1 Score"]
x         = np.arange(len(metrics))
width     = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

for i, (name, color) in enumerate(zip(MODEL_NAMES, PALETTE)):
    vals = [clf_results[name][m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color,
                  edgecolor="white", linewidth=0.6)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Fig. 1 — Classification Model Performance Comparison\n"
             "(Random Forest vs XGBoost vs Decision Tree)", fontweight="bold")
ax.legend(frameon=False)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("fig1_classification_comparison.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 2 ─ Regression Metrics Grouped Bar Chart
# ─────────────────────────────────────────────────────────────

reg_metrics = ["R² Score", "RMSE", "MAE"]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, metric in zip(axes, reg_metrics):
    vals   = [reg_results[name][metric] for name in MODEL_NAMES]
    colors = PALETTE
    bars   = ax.bar(MODEL_NAMES, vals, color=colors, edgecolor="white", linewidth=0.6)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.02,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(metric, fontweight="bold")
    ax.set_xticklabels(MODEL_NAMES, rotation=12, ha="right")
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle("Fig. 2 — Regression Model Performance Comparison\n"
             "(R², RMSE, MAE)", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig2_regression_comparison.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 3 ─ Confusion Matrices (all 3 classifiers side by side)
# ─────────────────────────────────────────────────────────────

risk_labels = le_risk.classes_   # e.g. ['High Risk', 'Low Risk', 'Medium Risk']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, color) in zip(axes, zip(MODEL_NAMES, PALETTE)):
    cm = confusion_matrix(y_test_c, clf_preds[name])
    sns.heatmap(cm, annot=True, fmt="d", ax=ax,
                xticklabels=risk_labels, yticklabels=risk_labels,
                cmap=sns.light_palette(color, as_cmap=True),
                linewidths=0.5, linecolor="white",
                cbar=False, annot_kws={"size": 10})
    ax.set_title(f"{name}", fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.tick_params(axis="x", rotation=20)
    ax.tick_params(axis="y", rotation=0)

fig.suptitle("Fig. 3 — Confusion Matrices for Risk Level Classification",
             fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig("fig3_confusion_matrices.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 4 ─ Feature Importance (Random Forest — best model)
# ─────────────────────────────────────────────────────────────

feature_names = ["Year", "Station", "Zone", "Prev Year Accidents",
                 "Year-on-Year Trend", "2-Year Rolling Avg"]

rf_clf  = clf_models["Random Forest"]
rf_reg  = reg_models["Random Forest"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, title in zip(
    axes,
    [rf_clf, rf_reg],
    ["Classification (Risk Level)", "Regression (Accident Count)"]
):
    importances = model.feature_importances_
    indices     = np.argsort(importances)
    colors_fi   = plt.cm.Blues(np.linspace(0.4, 0.9, len(indices)))

    ax.barh([feature_names[i] for i in indices], importances[indices],
            color=colors_fi, edgecolor="white")
    ax.set_xlabel("Importance Score")
    ax.set_title(f"Feature Importance — {title}", fontweight="bold")
    ax.xaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle("Fig. 4 — Random Forest Feature Importance Analysis",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig4_feature_importance.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 5 ─ Actual vs Predicted (Regression — all 3 models)
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, color) in zip(axes, zip(MODEL_NAMES, PALETTE)):
    y_pred = reg_preds[name]
    ax.scatter(y_test_r, y_pred, alpha=0.5, color=color,
               s=25, edgecolors="none", label="Predictions")
    lims = [min(y_test_r.min(), y_pred.min()),
            max(y_test_r.max(), y_pred.max())]
    ax.plot(lims, lims, "k--", linewidth=1, label="Perfect Fit")
    ax.set_xlabel("Actual Accidents")
    ax.set_ylabel("Predicted Accidents")
    ax.set_title(f"{name}\nR² = {reg_results[name]['R² Score']}", fontweight="bold")
    ax.legend(fontsize=8, frameon=False)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)

fig.suptitle("Fig. 5 — Actual vs Predicted Accident Count (Regression Models)",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig5_actual_vs_predicted.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 6 ─ Year-wise Accident Trend by Zone
# ─────────────────────────────────────────────────────────────

year_cols = ["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"]

# Use your existing 'df' (wide format with Zone column)
zone_trend = df.groupby("Zone")[year_cols].sum()

fig, ax = plt.subplots(figsize=(10, 5))

zone_palette = plt.cm.tab10.colors
years = [int(y.replace("Y","")) for y in year_cols]

for i, (zone, row) in enumerate(zone_trend.iterrows()):
    ax.plot(years, row.values, marker="o", linewidth=2,
            color=zone_palette[i % 10], label=zone, markersize=5)

ax.set_xlabel("Year")
ax.set_ylabel("Total Accidents")
ax.set_title("Fig. 6 — Year-wise Accident Trend by Zone (2018–2025)",
             fontweight="bold")
ax.legend(title="Zone", frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
ax.set_xticks(years)
plt.tight_layout()
plt.savefig("fig6_zone_trend.png", bbox_inches="tight")
plt.show()
print("\n")



## 🧠 SHAP Explainability Analysis

In [ ]:
# ============================================================
# CELL 3 — SHAP EXPLAINABILITY ANALYSIS
# ============================================================

# !pip install shap -q   ← uncomment first time only

import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      11,
    "axes.titlesize": 13,
    "figure.dpi":     150,
})

feature_names = [
    "Year",
    "Station",
    "Zone",
    "Prev Year Accidents",
    "Year-on-Year Trend",
    "2-Year Rolling Avg"
]

# ── Convert test sets to DataFrames ──────────────────────────
X_test_c_df = pd.DataFrame(np.array(X_test_c), columns=feature_names)
X_test_r_df = pd.DataFrame(np.array(X_test_r), columns=feature_names)

# ── Best model = Random Forest ───────────────────────────────
best_clf = clf_models["Random Forest"]
best_reg = reg_models["Random Forest"]

# ── SHAP Explainers ───────────────────────────────────────────
print("Computing SHAP values (may take ~30 seconds)...")

explainer_clf = shap.TreeExplainer(best_clf)
explainer_reg = shap.TreeExplainer(best_reg)

shap_values_clf = explainer_clf.shap_values(X_test_c_df)
shap_values_reg = explainer_reg.shap_values(X_test_r_df)

# ── Handle multi-class shape: list or 3D array ───────────────
if isinstance(shap_values_clf, np.ndarray) and shap_values_clf.ndim == 3:
    n_classes     = shap_values_clf.shape[2]
    shap_clf_list = [shap_values_clf[:, :, i] for i in range(n_classes)]
elif isinstance(shap_values_clf, list):
    shap_clf_list = shap_values_clf
    n_classes     = len(shap_clf_list)
else:
    shap_clf_list = [shap_values_clf]
    n_classes     = 1

risk_labels_display = le_risk.classes_
print(f"SHAP values computed! Classes: {list(risk_labels_display)}\n")

# ─────────────────────────────────────────────────────────────
# FIG 7A — SHAP Summary Plot per Risk Class
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, n_classes, figsize=(5 * n_classes, 5))
if n_classes == 1:
    axes = [axes]

for i, (ax, label) in enumerate(zip(axes, risk_labels_display)):
    plt.sca(ax)
    shap.summary_plot(
        shap_clf_list[i],
        X_test_c_df,
        show=False,
        plot_size=None,
        color_bar=(i == n_classes - 1)
    )
    ax.set_title(f"{label}", fontweight="bold")

fig.suptitle(
    "Fig. 7A — SHAP Summary: Feature Impact on Risk Level Classification",
    fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("fig7a_shap_classification_summary.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 7B — SHAP Summary Plot: Regression
# ─────────────────────────────────────────────────────────────

plt.figure(figsize=(8, 5))
shap.summary_plot(shap_values_reg, X_test_r_df, show=False)
plt.title(
    "Fig. 7B — SHAP Summary: Feature Impact on Accident Count Prediction",
    fontweight="bold"
)
plt.tight_layout()
plt.savefig("fig7b_shap_regression_summary.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 8 — Mean SHAP Bar Chart (Both Tasks)
# ─────────────────────────────────────────────────────────────

mean_shap_clf = np.mean([np.abs(sv).mean(axis=0) for sv in shap_clf_list], axis=0)
mean_shap_reg = np.abs(shap_values_reg).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, mean_shap, title, cmap_name in zip(
    axes,
    [mean_shap_clf, mean_shap_reg],
    ["Classification — Risk Level", "Regression — Accident Count"],
    ["Blues", "Oranges"]
):
    indices    = np.argsort(mean_shap)
    bar_colors = getattr(plt.cm, cmap_name)(np.linspace(0.4, 0.9, len(indices)))
    ax.barh(
        [feature_names[i] for i in indices],
        mean_shap[indices],
        color=bar_colors, edgecolor="white"
    )
    ax.set_xlabel("Mean |SHAP Value|")
    ax.set_title(title, fontweight="bold")
    ax.xaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

fig.suptitle(
    "Fig. 8 — Mean SHAP Feature Importance (Classification vs Regression)",
    fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("fig8_shap_mean_importance.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 9 — SHAP Waterfall Plot (Single Prediction Explained)
# ─────────────────────────────────────────────────────────────

sample_idx = 0

ev_reg = explainer_reg.expected_value
if isinstance(ev_reg, (list, np.ndarray)):
    ev_reg = float(np.array(ev_reg).flatten()[0])

shap_exp_reg = shap.Explanation(
    values        = shap_values_reg[sample_idx],
    base_values   = ev_reg,
    data          = X_test_r_df.iloc[sample_idx].values,
    feature_names = feature_names
)

plt.figure(figsize=(8, 4))
shap.waterfall_plot(shap_exp_reg, show=False)
plt.title(
    "Fig. 9 — SHAP Waterfall: Why the Model Predicted This Accident Count",
    fontweight="bold"
)
plt.tight_layout()
plt.savefig("fig9_shap_waterfall.png", bbox_inches="tight")
plt.show()

# ── Key Insights ─────────────────────────────────────────────
print("\nSHAP Analysis complete! Figures saved: fig7a, fig7b, fig8, fig9")
print("\nKey Insights for your paper:")
print(f"   Most influential for Regression    : {feature_names[np.argmax(mean_shap_reg)]}")
print(f"   Most influential for Classification: {feature_names[np.argmax(mean_shap_clf)]}")

## ⚖️ Class Imbalance Check & SMOTE

In [ ]:
# ============================================================
# CELL 4 — CLASS IMBALANCE CHECK + FIX
# ============================================================

from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np

# !pip install imbalanced-learn -q

# ─────────────────────────────────────────────────────────────
# STEP 1 — Check class distribution BEFORE fix
# ─────────────────────────────────────────────────────────────

counter_before = Counter(y_train_c)
labels         = le_risk.classes_
counts_before  = [counter_before[i] for i in range(len(labels))]

print("Class Distribution BEFORE balancing:")
for label, count in zip(labels, counts_before):
    print(f"   {label:<15}: {count} samples")

total = sum(counts_before)
imbalance_ratio = max(counts_before) / min(counts_before)
print(f"\n   Imbalance Ratio (max/min): {imbalance_ratio:.2f}")

if imbalance_ratio > 1.5:
    print("   ⚠️  Imbalance detected — applying SMOTE")
else:
    print("   ✅  Classes are balanced — SMOTE not needed")

# ─────────────────────────────────────────────────────────────
# STEP 2 — Apply SMOTE if needed
# ─────────────────────────────────────────────────────────────

if imbalance_ratio > 1.5:
    smote = SMOTE(random_state=42)
    X_train_c_bal, y_train_c_bal = smote.fit_resample(X_train_c, y_train_c)
else:
    X_train_c_bal, y_train_c_bal = X_train_c, y_train_c

counter_after  = Counter(y_train_c_bal)
counts_after   = [counter_after[i] for i in range(len(labels))]

print("\nClass Distribution AFTER balancing:")
for label, count in zip(labels, counts_after):
    print(f"   {label:<15}: {count} samples")

# ─────────────────────────────────────────────────────────────
# FIG 10 — Class Distribution Before vs After
# ─────────────────────────────────────────────────────────────

x      = np.arange(len(labels))
width  = 0.35
colors = ["#2C6E91", "#E07B3F"]

fig, ax = plt.subplots(figsize=(8, 5))

bars1 = ax.bar(x - width/2, counts_before, width,
               label="Before SMOTE", color=colors[0], edgecolor="white")
bars2 = ax.bar(x + width/2, counts_after,  width,
               label="After SMOTE",  color=colors[1], edgecolor="white")

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1,
            str(int(bar.get_height())),
            ha="center", va="bottom", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Number of Samples")
ax.set_title("Fig. 10 — Class Distribution Before vs After SMOTE",
             fontweight="bold")
ax.legend(frameon=False)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("fig10_class_balance.png", bbox_inches="tight")
plt.show()

# ─────────────────────────────────────────────────────────────
# STEP 3 — Retrain RF Classifier on balanced data
# ─────────────────────────────────────────────────────────────

rf_balanced = RandomForestClassifier(n_estimators=200, random_state=42)
rf_balanced.fit(X_train_c_bal, y_train_c_bal)
y_pred_bal = rf_balanced.predict(X_test_c)

print("\n" + "="*50)
print("RF Classification Report — AFTER Balancing:")
print("="*50)
print(classification_report(y_test_c, y_pred_bal,
                             target_names=labels))

# ─────────────────────────────────────────────────────────────
# STEP 4 — Compare accuracy before vs after
# ─────────────────────────────────────────────────────────────

from sklearn.metrics import accuracy_score, f1_score

acc_before = clf_results["Random Forest"]["Accuracy"]
acc_after  = accuracy_score(y_test_c, y_pred_bal)
f1_before  = clf_results["Random Forest"]["F1 Score"]
f1_after   = f1_score(y_test_c, y_pred_bal, average="weighted")

print(f"\nAccuracy  — Before: {acc_before:.4f}  |  After: {acc_after:.4f}")
print(f"F1 Score  — Before: {f1_before:.4f}  |  After: {f1_after:.4f}")

if f1_after >= f1_before:
    print("\n✅ Balanced model is better or equal — use this for final results!")
    clf_models["Random Forest"] = rf_balanced
else:
    print("\n📌 Original model was better — keeping original RF for final results.")

## 🔧 Hyperparameter Tuning

In [ ]:
# ============================================================
# CELL 5 — HYPERPARAMETER TUNING (Random Forest — Best Model)
# ============================================================

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, f1_score,
                             r2_score, mean_squared_error, mean_absolute_error)
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      11,
    "axes.titlesize": 13,
    "figure.dpi":     150,
})

# ─────────────────────────────────────────────────────────────
# PARAM GRID
# ─────────────────────────────────────────────────────────────

param_grid = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2", None]
}

# ─────────────────────────────────────────────────────────────
# TUNE CLASSIFIER (on SMOTE-balanced data)
# ─────────────────────────────────────────────────────────────

print("Tuning RF Classifier (this may take ~1-2 mins)...")

search_clf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=30,
    scoring="f1_weighted",
    cv=5,
    random_state=42,
    n_jobs=-1
)
search_clf.fit(X_train_c_bal, y_train_c_bal)

best_clf_tuned = search_clf.best_estimator_
y_pred_clf_tuned = best_clf_tuned.predict(X_test_c)

acc_tuned_clf = accuracy_score(y_test_c, y_pred_clf_tuned)
f1_tuned_clf  = f1_score(y_test_c, y_pred_clf_tuned, average="weighted")

print(f"✅ Best Params (Classifier): {search_clf.best_params_}")
print(f"   Accuracy : {acc_tuned_clf:.4f}")
print(f"   F1 Score : {f1_tuned_clf:.4f}")

# ─────────────────────────────────────────────────────────────
# TUNE REGRESSOR
# ─────────────────────────────────────────────────────────────

print("\nTuning RF Regressor (this may take ~1-2 mins)...")

search_reg = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=30,
    scoring="r2",
    cv=5,
    random_state=42,
    n_jobs=-1
)
search_reg.fit(X_train_r, y_train_r)

best_reg_tuned = search_reg.best_estimator_
y_pred_reg_tuned = best_reg_tuned.predict(X_test_r)

r2_tuned   = r2_score(y_test_r, y_pred_reg_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test_r, y_pred_reg_tuned))
mae_tuned  = mean_absolute_error(y_test_r, y_pred_reg_tuned)

print(f"✅ Best Params (Regressor): {search_reg.best_params_}")
print(f"   R²   : {r2_tuned:.4f}")
print(f"   RMSE : {rmse_tuned:.4f}")
print(f"   MAE  : {mae_tuned:.4f}")

# ─────────────────────────────────────────────────────────────
# FIG 11 — Before vs After Tuning Comparison
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Classification ──
clf_metrics   = ["Accuracy", "F1 Score"]
before_clf    = [acc_after, f1_after]       # from SMOTE cell
after_clf     = [acc_tuned_clf, f1_tuned_clf]
x             = np.arange(len(clf_metrics))
width         = 0.35

ax = axes[0]
b1 = ax.bar(x - width/2, before_clf, width, label="Before Tuning",
            color="#2C6E91", edgecolor="white")
b2 = ax.bar(x + width/2, after_clf,  width, label="After Tuning",
            color="#3BAD6B", edgecolor="white")
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{bar.get_height():.4f}",
            ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(clf_metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Classification — Before vs After Tuning", fontweight="bold")
ax.legend(frameon=False)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

# ── Regression ──
reg_metrics = ["R²", "RMSE", "MAE"]
before_reg  = [reg_results["Random Forest"]["R² Score"],
               reg_results["Random Forest"]["RMSE"],
               reg_results["Random Forest"]["MAE"]]
after_reg   = [r2_tuned, rmse_tuned, mae_tuned]
x2          = np.arange(len(reg_metrics))

ax2 = axes[1]
b3  = ax2.bar(x2 - width/2, before_reg, width, label="Before Tuning",
              color="#2C6E91", edgecolor="white")
b4  = ax2.bar(x2 + width/2, after_reg,  width, label="After Tuning",
              color="#3BAD6B", edgecolor="white")
for bar in list(b3) + list(b4):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f"{bar.get_height():.2f}",
             ha="center", va="bottom", fontsize=8)
ax2.set_xticks(x2)
ax2.set_xticklabels(reg_metrics)
ax2.set_ylabel("Score")
ax2.set_title("Regression — Before vs After Tuning", fontweight="bold")
ax2.legend(frameon=False)
ax2.yaxis.grid(True, linestyle="--", alpha=0.5)
ax2.set_axisbelow(True)

fig.suptitle("Fig. 11 — Hyperparameter Tuning: Before vs After (Random Forest)",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig11_tuning_comparison.png", bbox_inches="tight")
plt.show()

# ─────────────────────────────────────────────────────────────
# Update best models for use in rest of notebook
# ─────────────────────────────────────────────────────────────

clf_models["Random Forest"] = best_clf_tuned
reg_models["Random Forest"] = best_reg_tuned

print("\n✅ Tuning complete! Best RF models updated.")
print("   Fig 11 saved: fig11_tuning_comparison.png")

## 🔗 Correlation Heatmap

In [ ]:
# ============================================================
# CELL 6 — CORRELATION HEATMAP
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      11,
    "axes.titlesize": 13,
    "figure.dpi":     150,
})

df = pd.read_csv("data/final_perfect_dataset.csv")

year_cols = ["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"]
# Replace 0 with NaN and remove rows with no data for consistency with model training
df[year_cols] = df[year_cols].replace(0, np.nan)
df = df[df[year_cols].notna().sum(axis=1) > 0]

# Convert to long format
df_long = df.melt(
    id_vars=["Station", "Zone", "Latitude", "Longitude"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Total"
)
df_long["Year"] = df_long["Year"].str.replace("Y"," ").astype(int)

# Feature Engineering (as done in UfxAJ9MkNUAg)
df_long = df_long.sort_values(["Station", "Year"])
df_long["Prev_Year"] = df_long.groupby("Station")["Total"].shift(1)
df_long["Trend"] = df_long["Total"] - df_long["Prev_Year"]
df_long["Rolling_Avg"] = df_long.groupby("Station")["Total"].rolling(2).mean().reset_index(0, drop=True)
df_long = df_long.fillna(0)

# Create Risk Level (as done in UfxAJ9MkNUAg)
conditions = [
    (df_long["Total"] < 30),
    (df_long["Total"] < 70),
    (df_long["Total"] >= 70)
]
labels = ["Low Risk", "Medium Risk", "High Risk"]
df_long["Risk_Level"] = np.select(conditions, labels, default="")

# Encoding (as done in UfxAJ9MkNUAg)
le_station = LabelEncoder()
le_zone = LabelEncoder()
le_risk = LabelEncoder()
df_long["Station_Enc"] = le_station.fit_transform(df_long["Station"])
df_long["Zone_Enc"] = le_zone.fit_transform(df_long["Zone"])
df_long["Risk_Enc"] = le_risk.fit_transform(df_long["Risk_Level"])

# ─────────────────────────────────────────────────────────────
# STEP 1 — Build correlation dataframe from df_long
# ─────────────────────────────────────────────────────────────

corr_df = df_long[[
    "Year",
    "Total",
    "Prev_Year",
    "Trend",
    "Rolling_Avg",
    "Station_Enc",
    "Zone_Enc",
    "Risk_Enc"
]].copy()

corr_df.columns = [
    "Year",
    "Accident Count",
    "Prev Year Accidents",
    "Year-on-Year Trend",
    "2-Year Rolling Avg",
    "Station",
    "Zone",
    "Risk Level"
]

corr_matrix = corr_df.corr()

# ─────────────────────────────────────────────────────────────
# FIG 12A — Full Correlation Heatmap
# ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 7))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    linecolor="white",
    square=True,
    ax=ax,
    cbar_kws={"shrink": 0.8},
    annot_kws={"size": 9}
)

ax.set_title(
    "Fig. 12A — Feature Correlation Heatmap\n"
    "(Bangalore Road Accident Dataset 2018–2025)",
    fontweight="bold"
)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig("fig12a_correlation_heatmap.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# FIG 12B — Correlation with Target (Accident Count)
# ─────────────────────────────────────────────────────────────

target_corr = corr_matrix["Accident Count"].drop("Accident Count").sort_values()

colors = ["#d73027" if v < 0 else "#2C6E91" for v in target_corr]

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.barh(target_corr.index, target_corr.values,
               color=colors, edgecolor="white")

for bar, val in zip(bars, target_corr.values):
    ax.text(
        val + (0.01 if val >= 0 else -0.01),
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}",
        va="center",
        ha="left" if val >= 0 else "right",
        fontsize=9
    )

ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Pearson Correlation Coefficient")
ax.set_title(
    "Fig. 12B — Feature Correlation with Accident Count",
    fontweight="bold"
)
ax.xaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("fig12b_target_correlation.png", bbox_inches="tight")
plt.show()
print("\n")

# ─────────────────────────────────────────────────────────────
# Print key correlations
# ─────────────────────────────────────────────────────────────

print("Correlation with Accident Count (sorted):")
print(target_corr.to_string())
print("\n✅ Figures saved: fig12a, fig12b")

## 🗺️ Upgraded Folium Map

In [ ]:
# ============================================================
# CELL 7 — UPGRADED FOLIUM MAP
# ============================================================

import folium
from folium.plugins import HeatMap, MarkerCluster
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# --- Start: Re-creating df_long with engineered features and risk levels ---
# Load final perfect dataset (df) and define year_cols
df = pd.read_csv("data/final_perfect_dataset.csv")

year_cols = ["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"]

# Replace 0 with NaN and remove rows with no data for consistency
df[year_cols] = df[year_cols].replace(0, np.nan)
df = df[df[year_cols].notna().sum(axis=1) > 0]

# Convert to long format
df_long = df.melt(
    id_vars=["Station", "Zone", "Latitude", "Longitude"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Total"
)
df_long["Year"] = df_long["Year"].str.replace("Y"," ").astype(int)

# Feature Engineering (as done in UfxAJ9MkNUAg and YJeJru-E_R9M)
df_long = df_long.sort_values(["Station", "Year"])
df_long["Prev_Year"] = df_long.groupby("Station")["Total"].shift(1)
df_long["Trend"] = df_long["Total"] - df_long["Prev_Year"]
df_long["Rolling_Avg"] = df_long.groupby("Station")["Total"].rolling(2).mean().reset_index(0, drop=True)
df_long = df_long.fillna(0) # Fill NaN from shift/rolling_avg for consistent data

# Create Risk Level (as done in UfxAJ9MkNUAg and YJeJru-E_R9M)
conditions = [
    (df_long["Total"] < 30),
    (df_long["Total"] < 70),
    (df_long["Total"] >= 70)
]
labels = ["Low Risk", "Medium Risk", "High Risk"]
df_long["Risk_Level"] = np.select(conditions, labels, default="")
# --- End: Re-creating df_long ---


# ─────────────────────────────────────────────────────────────
# STEP 1 — Prepare station-level summary from df_long
# ─────────────────────────────────────────────────────────────

# Get latest year data per station
latest_year = df_long["Year"].max()

station_summary = df_long.groupby("Station").agg(
    Zone          = ("Zone", "first"),
    Latitude      = ("Latitude", "first"),
    Longitude     = ("Longitude", "first"),
    Total_Avg     = ("Total", "mean"),
    Total_Sum     = ("Total", "sum"),
    Risk_Level    = ("Risk_Level", lambda x: x.mode()[0])
).reset_index()

# Drop stations with missing coordinates
station_summary = station_summary.dropna(subset=["Latitude", "Longitude"])

# ─────────────────────────────────────────────────────────────
# STEP 2 — Risk color mapping
# ─────────────────────────────────────────────────────────────

def get_color(risk):
    return {
        "High Risk":   "#d73027",
        "Medium Risk": "#fc8d59",
        "Low Risk":    "#1a9850"
    }.get(risk, "#999999")

def get_radius(total_avg):
    # Scale circle size by average accidents
    return max(8, min(30, total_avg / 5))

# ─────────────────────────────────────────────────────────────
# STEP 3 — Build Map with Multiple Layers
# ─────────────────────────────────────────────────────────────

bangalore_map = folium.Map(
    location  = [12.9716, 77.5946],
    zoom_start = 11,
    tiles      = "CartoDB positron"
)

# ── Layer 1: Risk Level Circle Markers ───────────────────────
risk_layer = folium.FeatureGroup(name="🔴 Risk Level Markers", show=True)

for _, row in station_summary.iterrows():
    color  = get_color(row["Risk_Level"])
    radius = get_radius(row["Total_Avg"])

    tooltip_text = (
        f"<b>{row['Station']}</b><br>"
        f"Zone: {row['Zone']}<br>"
        f"Risk Level: <b style='color:{color}'>{row['Risk_Level']}</b><br>"
        f"Avg Accidents/Year: {row['Total_Avg']:.1f}<br>"
        f"Total (2018–2025): {int(row['Total_Sum'])}"
    )

    folium.CircleMarker(
        location     = [row["Latitude"], row["Longitude"]],
        radius       = radius,
        color        = color,
        fill         = True,
        fill_color   = color,
        fill_opacity = 0.7,
        weight       = 2,
        tooltip      = folium.Tooltip(tooltip_text, sticky=True)
    ).add_to(risk_layer)

risk_layer.add_to(bangalore_map)

# ── Layer 2: HeatMap ─────────────────────────────────────────
heat_layer = folium.FeatureGroup(name="🌡️ Accident Heatmap", show=False)

heat_data = station_summary[["Latitude", "Longitude", "Total_Avg"]].dropna()
heat_list = heat_data.values.tolist()

HeatMap(
    heat_list,
    radius    = 30,
    blur      = 20,
    max_zoom  = 13,
    gradient  = {
        "0.2": "#1a9850",
        "0.5": "#fc8d59",
        "0.8": "#d73027",
        "1.0": "#800026"
    }
).add_to(heat_layer)

heat_layer.add_to(bangalore_map)

# ── Layer 3: Top 10 Hotspot Markers ──────────────────────────
hotspot_layer = folium.FeatureGroup(name="📍 Top 10 Hotspots", show=True)

top10 = station_summary.nlargest(10, "Total_Sum")

for rank, (_, row) in enumerate(top10.iterrows(), 1):
    folium.Marker(
        location  = [row["Latitude"], row["Longitude"]],
        tooltip   = folium.Tooltip(
            f"<b>#{rank} {row['Station']}</b><br>"
            f"Total Accidents: {int(row['Total_Sum'])}<br>"
            f"Risk: {row['Risk_Level']}",
            sticky=True
        ),
        icon = folium.DivIcon(html=f"""
            <div style="
                background:#1d3557;
                color:white;
                border-radius:50%;
                width:26px;
                height:26px;
                display:flex;
                align-items:center;
                justify-content:center;
                font-weight:bold;
                font-size:11px;
                border:2px solid white;
                box-shadow:1px 1px 4px rgba(0,0,0,0.4);">
                {rank}
            </div>""",
            icon_size=(26, 26),
            icon_anchor=(13, 13)
        )
    ).add_to(hotspot_layer)

hotspot_layer.add_to(bangalore_map)

# ─────────────────────────────────────────────────────────────
# STEP 4 — Legend
# ─────────────────────────────────────────────────────────────

legend_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    z-index: 1000;
    background: white;
    padding: 14px 18px;
    border-radius: 10px;
    box-shadow: 2px 2px 8px rgba(0,0,0,0.3);
    font-family: serif;
    font-size: 13px;
    line-height: 1.8;
">
    <b style="font-size:14px;">Risk Level Legend</b><br>
    <span style="color:#d73027;">●</span> High Risk<br>
    <span style="color:#fc8d59;">●</span> Medium Risk<br>
    <span style="color:#1a9850;">●</span> Low Risk<br>
    <hr style="margin:6px 0;">
    <span style="font-size:11px;color:#555;">
        Circle size ∝ Avg Accidents/Year<br>
        Data: Bangalore 2018–2025
    </span>
</div>
"""
bangalore_map.get_root().html.add_child(folium.Element(legend_html))

# ─────────────────────────────────────────────────────────────
# STEP 5 — Layer Control + Save
# ─────────────────────────────────────────────────────────────

folium.LayerControl(collapsed=False).add_to(bangalore_map)

bangalore_map.save("bangalore_accident_map_final.html")


print("✅ Upgraded map saved: bangalore_accident_map_final.html")
print(f"\n📌 Map Summary:")
print(f"   Total stations mapped : {len(station_summary)}")
print(f"   High Risk stations    : {(station_summary['Risk_Level'] == 'High Risk').sum()}")
print(f"   Medium Risk stations  : {(station_summary['Risk_Level'] == 'Medium Risk').sum()}")
print(f"   Low Risk stations     : {(station_summary['Risk_Level'] == 'Low Risk').sum()}")
print(f"\n   Top 3 Hotspots:")
for rank, (_, row) in enumerate(top10.head(3).iterrows(), 1):
    print(f"   #{rank} {row['Station']} — {int(row['Total_Sum'])} total accidents")

## 🚀 Final Optimized Model + Prediction Menu
*Incorporates SMOTE + Hyperparameter Tuning*

In [ ]:
# ============================================================
# MAIN MODEL CELL (UPGRADED) — Final Version
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, r2_score, mean_squared_error, mean_absolute_error
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# STEP 1 — Load & Prepare Data
# ─────────────────────────────────────────────────────────────

df = pd.read_csv("data/final_perfect_dataset.csv")

df_long = df.melt(
    id_vars   = ["Station", "Zone", "Latitude", "Longitude"],
    value_vars = ["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"],
    var_name  = "Year",
    value_name = "Total"
)

df_long["Year"] = df_long["Year"].str.replace("Y", "").astype(int)

# ─────────────────────────────────────────────────────────────
# STEP 2 — Feature Engineering
# ─────────────────────────────────────────────────────────────

df_long = df_long.sort_values(["Station", "Year"])

df_long["Prev_Year"]   = df_long.groupby("Station")["Total"].shift(1)
df_long["Trend"]       = df_long["Total"] - df_long["Prev_Year"]
df_long["Rolling_Avg"] = (df_long.groupby("Station")["Total"]
                          .rolling(2).mean().reset_index(0, drop=True))
df_long = df_long.fillna(0)

# ─────────────────────────────────────────────────────────────
# STEP 3 — Risk Level Labels
# ─────────────────────────────────────────────────────────────

conditions = [
    (df_long["Total"] < 30),
    (df_long["Total"] < 70),
    (df_long["Total"] >= 70)
]
risk_labels = ["Low Risk", "Medium Risk", "High Risk"]
df_long["Risk_Level"] = np.select(conditions, risk_labels, default="")

# ─────────────────────────────────────────────────────────────
# STEP 4 — Encoding
# ─────────────────────────────────────────────────────────────

le_station = LabelEncoder()
le_zone    = LabelEncoder()
le_risk    = LabelEncoder()

df_long["Station_Enc"] = le_station.fit_transform(df_long["Station"])
df_long["Zone_Enc"]    = le_zone.fit_transform(df_long["Zone"])
df_long["Risk_Enc"]    = le_risk.fit_transform(df_long["Risk_Level"])

# ─────────────────────────────────────────────────────────────
# STEP 5 — Train/Test Split
# ─────────────────────────────────────────────────────────────

features = ["Year", "Station_Enc", "Zone_Enc", "Prev_Year", "Trend", "Rolling_Avg"]

X_reg = df_long[features]
y_reg = df_long["Total"]

X_clf = df_long[features]
y_clf = df_long["Risk_Enc"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)

# ─────────────────────────────────────────────────────────────
# STEP 6 — SMOTE (Class Balancing)
# ─────────────────────────────────────────────────────────────

smote = SMOTE(random_state=42)
X_train_c_bal, y_train_c_bal = smote.fit_resample(X_train_c, y_train_c)

print("✅ SMOTE applied — balanced class distribution:")
from collections import Counter
for label, count in zip(le_risk.classes_, Counter(y_train_c_bal).values()):
    print(f"   {label:<15}: {count} samples")

# ─────────────────────────────────────────────────────────────
# STEP 7 — Hyperparameter Tuning
# ─────────────────────────────────────────────────────────────

param_grid = {
    "n_estimators":      [100, 200, 300, 500],
    "max_depth":         [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2", None]
}

print("\n⏳ Tuning Classifier...")
search_clf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions = param_grid,
    n_iter=30, scoring="f1_weighted",
    cv=5, random_state=42, n_jobs=-1
)
search_clf.fit(X_train_c_bal, y_train_c_bal)
clf_model = search_clf.best_estimator_

print("⏳ Tuning Regressor...")
search_reg = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions = param_grid,
    n_iter=30, scoring="r2",
    cv=5, random_state=42, n_jobs=-1
)
search_reg.fit(X_train_r, y_train_r)
reg_model = search_reg.best_estimator_

# ─────────────────────────────────────────────────────────────
# STEP 8 — Evaluate Final Models
# ─────────────────────────────────────────────────────────────

y_pred_c = clf_model.predict(X_test_c)
y_pred_r = reg_model.predict(X_test_r)

print("\n" + "="*50)
print("FINAL MODEL RESULTS")
print("="*50)

print("\n🔹 Classification (Risk Level):")
print(classification_report(y_test_c, y_pred_c, target_names=le_risk.classes_))

r2   = r2_score(y_test_r, y_pred_r)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
mae  = mean_absolute_error(y_test_r, y_pred_r)

print("🔹 Regression (Accident Count):")
print(f"   R²   : {r2:.4f}")
print(f"   RMSE : {rmse:.4f}")
print(f"   MAE  : {mae:.4f}")

# ─────────────────────────────────────────────────────────────
# STEP 9 — Interactive Prediction Menu
# ─────────────────────────────────────────────────────────────

stations = sorted(df["Station"].unique())

print("\n📍 Available Stations:")
for s in stations:
    print("-", s)

while True:
    print("\n===== MENU =====")
    print("1. Predict Accident Count")
    print("2. Predict Risk Level")
    print("3. Exit")

    choice = input("Enter your choice: ")

    if choice in ["1", "2"]:

        station_input = input("Enter Station Name: ")

        if station_input not in stations:
            print("❌ Invalid station!")
            continue

        try:
            year_input = int(input("Enter Year (e.g., 2026): "))
        except:
            print("❌ Invalid year!")
            continue

        station_enc  = le_station.transform([station_input])[0]
        zone_name    = df[df["Station"] == station_input]["Zone"].iloc[0]
        zone_enc     = le_zone.transform([zone_name])[0]

        station_data = df_long[df_long["Station"] == station_input].sort_values("Year")
        last_row     = station_data.iloc[-1]
        prev_year    = last_row["Total"]
        trend        = last_row["Total"] - station_data.iloc[-2]["Total"] \
                       if len(station_data) > 1 else 0
        rolling_avg  = station_data["Total"].tail(2).mean()

        user_data = pd.DataFrame({
            "Year":        [year_input],
            "Station_Enc": [station_enc],
            "Zone_Enc":    [zone_enc],
            "Prev_Year":   [prev_year],
            "Trend":       [trend],
            "Rolling_Avg": [rolling_avg]
        })

        if choice == "1":
            pred = reg_model.predict(user_data)
            print("\n✅ Prediction:")
            print(f"   Station           : {station_input}")
            print(f"   Year              : {year_input}")
            print(f"   Predicted Accidents: {int(pred[0])}")

        elif choice == "2":
            pred = clf_model.predict(user_data)
            risk = le_risk.inverse_transform(pred)[0]
            print("\n✅ Prediction:")
            print(f"   Station          : {station_input}")
            print(f"   Year             : {year_input}")
            print(f"   Predicted Risk   : {risk}")

    elif choice == "3":
        print("Exiting...")
        break

    else:
        print("❌ Invalid choice!")

## 🦠 COVID-19 Impact Analysis (2020–2021)

In [ ]:
# ============================================================
# IMPROVEMENT 1 — COVID IMPACT ANALYSIS (2020–2021 Dip & Recovery)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

df = pd.read_csv("data/final_perfect_dataset.csv")

year_cols = ["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"]
df[year_cols] = df[year_cols].replace(0, np.nan)

df_long = df.melt(
    id_vars=["Station", "Zone"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Total"
)
df_long["Year"] = df_long["Year"].str.replace("Y", "").astype(int)

yearly = df_long.groupby("Year")["Total"].sum().reset_index()
yearly_zone = df_long.groupby(["Year", "Zone"])["Total"].sum().reset_index()

# ── Compute % change vs 2019 baseline
baseline = yearly.loc[yearly["Year"] == 2019, "Total"].values[0]
yearly["Pct_Change"] = ((yearly["Total"] - baseline) / baseline) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("COVID-19 Impact on Bangalore Road Accidents (2018–2025)",
             fontsize=14, fontweight="bold", y=1.02)

# ─── Plot 1: Total accidents with COVID band ───────────────────
ax1 = axes[0]
colors = ["#d62728" if y in [2020, 2021] else "#1f77b4" for y in yearly["Year"]]
bars = ax1.bar(yearly["Year"], yearly["Total"], color=colors, edgecolor="white", width=0.6)
ax1.axvspan(2019.5, 2021.5, alpha=0.08, color="red", label="COVID Period")
ax1.set_title("Total Accidents — COVID Dip & Recovery", fontsize=11)
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Accidents")
ax1.set_xticks(yearly["Year"])
ax1.set_xticklabels(yearly["Year"], rotation=45)
for bar, val in zip(bars, yearly["Total"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f"{int(val):,}", ha="center", va="bottom", fontsize=8)
covid_patch = mpatches.Patch(color="red", alpha=0.3, label="COVID Period (2020–21)")
ax1.legend(handles=[covid_patch], fontsize=9)

# ─── Plot 2: % change vs 2019 baseline ─────────────────────────
ax2 = axes[1]
pct_colors = ["#2ca02c" if p >= 0 else "#d62728" for p in yearly["Pct_Change"]]
ax2.bar(yearly["Year"], yearly["Pct_Change"], color=pct_colors, edgecolor="white", width=0.6)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.axvspan(2019.5, 2021.5, alpha=0.08, color="red")
ax2.set_title("% Change vs 2019 Baseline", fontsize=11)
ax2.set_xlabel("Year")
ax2.set_ylabel("% Change")
ax2.set_xticks(yearly["Year"])
ax2.set_xticklabels(yearly["Year"], rotation=45)
for x, p in zip(yearly["Year"], yearly["Pct_Change"]):
    ax2.text(x, p + (1 if p >= 0 else -2.5), f"{p:.1f}%",
             ha="center", va="bottom", fontsize=8)

# ─── Plot 3: Zone-wise COVID comparison (2019 vs 2020 vs 2022) ──
ax3 = axes[2]
pivot = yearly_zone[yearly_zone["Year"].isin([2019, 2020, 2022])].pivot(
    index="Zone", columns="Year", values="Total"
).fillna(0)
pivot.plot(kind="bar", ax=ax3,
           color=["#1f77b4", "#d62728", "#2ca02c"],
           edgecolor="white", width=0.7)
ax3.set_title("Zone-wise: Pre-COVID vs COVID vs Recovery", fontsize=11)
ax3.set_xlabel("Zone")
ax3.set_ylabel("Total Accidents")
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=30, ha="right")
ax3.legend(title="Year", fontsize=9)

plt.tight_layout()
plt.savefig("covid_impact_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Print summary stats
dip_2020 = yearly.loc[yearly["Year"]==2020, "Pct_Change"].values[0]
dip_2021 = yearly.loc[yearly["Year"]==2021, "Pct_Change"].values[0]
rec_2022 = yearly.loc[yearly["Year"]==2022, "Pct_Change"].values[0]

print("\n📊 COVID Impact Summary")
print("=" * 40)
print(f"  2020 vs 2019 : {dip_2020:+.1f}%  ← Lockdown dip")
print(f"  2021 vs 2019 : {dip_2021:+.1f}%  ← Continued low")
print(f"  2022 vs 2019 : {rec_2022:+.1f}%  ← Recovery phase")
print("\n  Insight: COVID lockdowns caused a significant reduction in")
print("  road accidents. Post-2021 recovery reflects increased mobility.")


## 🎯 Prediction with Confidence & Probability

In [ ]:
# ============================================================
# IMPROVEMENT 2 — PREDICTION WITH CONFIDENCE / PROBABILITY
# ============================================================
# Run AFTER the main model cell (MAIN MODEL CELL) so clf_model,
# le_risk, le_station, le_zone, df, and df_long are all in scope.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def predict_with_confidence(station_input, year_input,
                             clf_model, reg_model,
                             le_station, le_zone, le_risk,
                             df, df_long,
                             verbose=True):
    """
    Returns a dict with predicted risk label, per-class probabilities,
    predicted accident count, and a confidence bar chart.
    """
    if station_input not in df["Station"].values:
        raise ValueError(f"Station '{station_input}' not found in dataset.")

    station_enc = le_station.transform([station_input])[0]
    zone_name   = df[df["Station"] == station_input]["Zone"].iloc[0]
    zone_enc    = le_zone.transform([zone_name])[0]

    station_data = df_long[df_long["Station"] == station_input].sort_values("Year")
    last_row     = station_data.iloc[-1]
    prev_year    = last_row["Total"]
    trend = (
        last_row["Total"] - station_data.iloc[-2]["Total"]
        if len(station_data) > 1 else 0
    )
    rolling_avg = station_data["Total"].tail(2).mean()

    user_data = pd.DataFrame({
        "Year":        [year_input],
        "Station_Enc": [station_enc],
        "Zone_Enc":    [zone_enc],
        "Prev_Year":   [prev_year],
        "Trend":       [trend],
        "Rolling_Avg": [rolling_avg]
    })

    # ── Classification with probabilities
    proba       = clf_model.predict_proba(user_data)[0]
    class_names = le_risk.classes_                    # decoded labels
    pred_idx    = np.argmax(proba)
    pred_label  = class_names[pred_idx]
    confidence  = proba[pred_idx] * 100

    # ── Regression
    pred_count  = int(reg_model.predict(user_data)[0])

    result = {
        "station":    station_input,
        "zone":       zone_name,
        "year":       year_input,
        "risk_label": pred_label,
        "confidence": confidence,
        "probabilities": dict(zip(class_names, proba * 100)),
        "predicted_accidents": pred_count
    }

    if verbose:
        # ── Console output
        print("\n" + "="*50)
        print(" ACCIDENT RISK PREDICTION WITH CONFIDENCE")
        print("="*50)
        print(f"  Station            : {station_input}")
        print(f"  Zone               : {zone_name}")
        print(f"  Year               : {year_input}")
        print(f"  Predicted Accidents: {pred_count}")
        print(f"  Predicted Risk     : {pred_label}  ({confidence:.1f}% confidence)")
        print("\n  Class Probabilities:")
        for label, prob in sorted(result["probabilities"].items(),
                                  key=lambda x: -x[1]):
            bar = "█" * int(prob / 5)
            print(f"    {label:<15}: {prob:5.1f}%  {bar}")

        # ── Probability bar chart
        risk_colors = {"High Risk": "#d62728",
                       "Medium Risk": "#ff7f0e",
                       "Low Risk": "#2ca02c"}

        labels_sorted = sorted(result["probabilities"].keys(),
                               key=lambda x: -result["probabilities"][x])
        probs_sorted  = [result["probabilities"][l] for l in labels_sorted]
        bar_colors    = [risk_colors.get(l, "#1f77b4") for l in labels_sorted]

        fig, ax = plt.subplots(figsize=(6, 3))
        bars = ax.barh(labels_sorted, probs_sorted,
                       color=bar_colors, edgecolor="white", height=0.5)
        ax.set_xlim(0, 100)
        ax.set_xlabel("Probability (%)")
        ax.set_title(
            f"Risk Prediction Confidence\n{station_input} — {year_input}",
            fontweight="bold"
        )
        for bar, prob in zip(bars, probs_sorted):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    f"{prob:.1f}%", va="center", fontsize=10, fontweight="bold")
        # Highlight winner
        winner_idx = labels_sorted.index(pred_label)
        bars[winner_idx].set_edgecolor("black")
        bars[winner_idx].set_linewidth(2)
        plt.tight_layout()
        plt.savefig(f"confidence_{station_input.replace(' ','_')}_{year_input}.png",
                    dpi=150, bbox_inches="tight")
        plt.show()

    return result


# ── Demo: run for a few stations
sample_stations = df["Station"].sort_values().unique()[:3].tolist()

for station in sample_stations:
    predict_with_confidence(
        station_input = station,
        year_input    = 2026,
        clf_model     = clf_model,
        reg_model     = reg_model,
        le_station    = le_station,
        le_zone       = le_zone,
        le_risk       = le_risk,
        df            = df,
        df_long       = df_long
    )

# ── Interactive prompt version (replaces old menu option 2)
print("\n" + "="*50)
print(" INTERACTIVE PREDICTION (with confidence)")
print("="*50)
station_input = input("Enter Station Name (or press Enter to skip): ").strip()
if station_input:
    try:
        year_input = int(input("Enter Year (e.g., 2026): "))
        predict_with_confidence(
            station_input, year_input,
            clf_model, reg_model,
            le_station, le_zone, le_risk,
            df, df_long
        )
    except Exception as e:
        print(f"Error: {e}")


## 📊 Year-over-Year Station Ranking

In [ ]:
# ============================================================
# IMPROVEMENT 3 — YEAR-OVER-YEAR STATION RANKING
# Which stations improved vs worsened over 8 years (2018–2025)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

df = pd.read_csv("data/final_perfect_dataset.csv")

year_cols = ["Y2018","Y2019","Y2020","Y2021","Y2022","Y2023","Y2024","Y2025"]
df[year_cols] = df[year_cols].replace(0, np.nan)

# ── For each station compute:
#    • absolute change  : Y2025 - Y2018
#    • % change         : (Y2025 - Y2018) / Y2018 * 100
#    • linear slope     : trend across all 8 years (positive = worsening)
#    • rank 2018 / 2025 : position by total accidents each year

df_rank = df[["Station", "Zone"] + year_cols].copy()

# Drop stations missing either endpoint
df_rank = df_rank.dropna(subset=["Y2018", "Y2025"])

df_rank["Abs_Change"]  = df_rank["Y2025"] - df_rank["Y2018"]
df_rank["Pct_Change"]  = (df_rank["Abs_Change"] / df_rank["Y2018"]) * 100

# Linear slope (accidents per year trend)
years_num = list(range(len(year_cols)))
def slope(row):
    vals = row[year_cols].values.astype(float)
    mask = ~np.isnan(vals)
    if mask.sum() < 3:
        return np.nan
    return np.polyfit(np.array(years_num)[mask], vals[mask], 1)[0]

df_rank["Trend_Slope"] = df_rank.apply(slope, axis=1)
df_rank = df_rank.dropna(subset=["Trend_Slope"])

# Rank by total accidents each year (1 = worst)
df_rank["Rank_2018"] = df_rank["Y2018"].rank(ascending=False).astype(int)
df_rank["Rank_2025"] = df_rank["Y2025"].rank(ascending=False).astype(int)
df_rank["Rank_Change"] = df_rank["Rank_2018"] - df_rank["Rank_2025"]  # positive = improved

# ── Top Improvers & Top Worseners (by % change, excluding COVID outliers)
top_improved  = df_rank.nsmallest(10, "Pct_Change")   # most negative = most improved
top_worsened  = df_rank.nlargest(10,  "Pct_Change")   # most positive = most worsened

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Year-over-Year Station Ranking: 2018 → 2025",
             fontsize=14, fontweight="bold", y=1.02)

# ─── Plot 1: Top 10 Most Improved ──────────────────────────────
ax1 = axes[0]
ax1.barh(top_improved["Station"][::-1],
         top_improved["Pct_Change"][::-1],
         color="#2ca02c", edgecolor="white")
ax1.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax1.set_title("Top 10 Most Improved Stations", fontsize=11)
ax1.set_xlabel("% Change in Accidents (2018→2025)")
for bar, val in zip(ax1.patches, top_improved["Pct_Change"][::-1]):
    ax1.text(bar.get_width() - 1, bar.get_y() + bar.get_height()/2,
             f"{val:.1f}%", va="center", ha="right",
             color="white", fontsize=8, fontweight="bold")

# ─── Plot 2: Top 10 Most Worsened ──────────────────────────────
ax2 = axes[1]
ax2.barh(top_worsened["Station"][::-1],
         top_worsened["Pct_Change"][::-1],
         color="#d62728", edgecolor="white")
ax2.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_title("Top 10 Most Worsened Stations", fontsize=11)
ax2.set_xlabel("% Change in Accidents (2018→2025)")
for bar, val in zip(ax2.patches, top_worsened["Pct_Change"][::-1]):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f"+{val:.1f}%", va="center", ha="left",
             color="#d62728", fontsize=8, fontweight="bold")

# ─── Plot 3: Rank Change Scatter (2018 rank vs 2025 rank) ──────
ax3 = axes[2]
scatter_colors = df_rank["Rank_Change"].apply(
    lambda x: "#2ca02c" if x > 2 else ("#d62728" if x < -2 else "#aec7e8")
)
ax3.scatter(df_rank["Rank_2018"], df_rank["Rank_2025"],
            c=scatter_colors, alpha=0.7, s=60, edgecolors="white", linewidth=0.4)
lims = [1, max(df_rank["Rank_2018"].max(), df_rank["Rank_2025"].max())]
ax3.plot(lims, lims, "k--", linewidth=1, label="No change")
ax3.set_title("Station Rank: 2018 vs 2025\n(below diagonal = improved)",
              fontsize=11)
ax3.set_xlabel("Rank in 2018 (1 = most accidents)")
ax3.set_ylabel("Rank in 2025")
improved_patch  = mpatches.Patch(color="#2ca02c", label="Improved (rank ↓ >2)")
worsened_patch  = mpatches.Patch(color="#d62728", label="Worsened (rank ↑ >2)")
stable_patch    = mpatches.Patch(color="#aec7e8", label="Stable")
ax3.legend(handles=[improved_patch, worsened_patch, stable_patch], fontsize=8)

plt.tight_layout()
plt.savefig("station_ranking_yoy.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Summary table
print("\n📊 Station Ranking Summary")
print("=" * 60)
improved_count = (df_rank["Abs_Change"] < 0).sum()
worsened_count = (df_rank["Abs_Change"] > 0).sum()
stable_count   = (df_rank["Abs_Change"] == 0).sum()
print(f"  Stations improved (fewer accidents) : {improved_count}")
print(f"  Stations worsened (more accidents)  : {worsened_count}")
print(f"  Stations stable                     : {stable_count}")

print("\n  Top 5 Most Improved (2018→2025):")
for _, r in top_improved.head(5).iterrows():
    print(f"    {r['Station']:<25}: {r['Pct_Change']:+.1f}%  "
          f"(Rank {r['Rank_2018']} → {r['Rank_2025']})")

print("\n  Top 5 Most Worsened (2018→2025):")
for _, r in top_worsened.head(5).iterrows():
    print(f"    {r['Station']:<25}: {r['Pct_Change']:+.1f}%  "
          f"(Rank {r['Rank_2018']} → {r['Rank_2025']})")

# ── Full sortable table
display_cols = ["Station", "Zone", "Y2018", "Y2025",
                "Abs_Change", "Pct_Change", "Rank_2018", "Rank_2025", "Rank_Change"]
summary_table = df_rank[display_cols].sort_values("Pct_Change")
summary_table.columns = ["Station", "Zone", "2018", "2025",
                          "Abs Δ", "% Δ", "Rank 2018", "Rank 2025", "Rank Δ"]
print("\n  Full Station Rankings (sorted best → worst):")
print(summary_table.to_string(index=False))
